# ⚡ Módulo 13 - Notebook 01: PySpark SQL, Window y DeltaLake

## 🗄️ SQL Distribuido y Almacenamiento Transaccional

**Libro:** Saliendo de lo Pandito  
**Módulo:** 13 - PySpark SQL Window DeltaLake  
**Duración estimada:** 80 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Ejecutar** consultas SQL distribuidas con spark.sql()  
✅ **Entender** Window Functions en PySpark  
✅ **Trabajar** con Delta Lake (formato ACID)  
✅ **Aplicar** Time Travel y versionado  
✅ **Optimizar** consultas SQL en cluster

---

## 📋 Pre-requisitos

* ✅ Módulos 11 y 12 completados (PySpark Core y Transformaciones)
* ✅ Conocimiento de SQL tradicional
* ✅ Familiaridad con Spark DataFrames

---

## 📚 Contenido

1. PySpark SQL: Consultas Declarativas
2. Spark SQL vs DataFrame API
3. Introducción a Window Functions
4. Delta Lake: Formato ACID
5. Time Travel y Versionado
6. Caso Integrador: Pipeline SQL + Delta

---

## 💡 Por qué importa

**SQL + Delta Lake = Poder empresarial:**

* 📊 **SQL:** Lenguaje universal para analistas
* ⚡ **Distribuido:** Escala a TB/PB de datos
* 🔒 **ACID:** Transacciones confiables
* 🕐 **Time Travel:** Auditoría y rollback

**El stack completo de datos modernos**

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market (SPARK DataFrame)
    df = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    
    # Registrar como vista temporal para usar SQL
    df.createOrReplaceTempView("ventas")
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros: {df.count():,}")
    print(f"   🗄️ Particiones: {df.rdd.getNumPartitions()}")
    print(f"   📍 Ubicación: Mendoza, Argentina (Los Andes Market)")
    
    print(f"\n📋 Vista SQL creada: 'ventas'")
    print(f"   Ahora puedes usar: spark.sql('SELECT * FROM ventas')")
    
    # Mostrar esquema
    print(f"\n📊 Esquema de la tabla:")
    df.printSchema()
    
    # Ejecutar query SQL de ejemplo
    print(f"\n🔍 Query SQL de ejemplo:")
    result = spark.sql("""
        SELECT 
            zona,
            COUNT(*) as registros,
            SUM(ventas) as ventas_totales
        FROM ventas
        GROUP BY zona
        ORDER BY ventas_totales DESC
    """)
    result.show()
    
    print(f"\n🎯 Este notebook usará SQL DISTRIBUIDO sobre datos REALES")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 PySpark SQL: El Mejor de Dos Mundos

### 🗄️ ¿Qué es PySpark SQL?

**PySpark SQL** permite ejecutar consultas SQL estándar sobre DataFrames distribuidos.

**Ventajas:**
* ✅ **Familiar:** Mismo SQL que conoces
* ✅ **Distribuido:** Escala a TB/PB
* ✅ **Optimizado:** Catalyst Optimizer
* ✅ **Integrado:** Combina con DataFrame API

---

### 🔄 Dos Formas de Hacer lo Mismo

**DataFrame API:**
```python
df.filter(F.col("ventas") > 100000) \
  .groupBy("zona") \
  .agg(F.sum("ventas").alias("total")) \
  .orderBy(F.desc("total"))
```

**SQL API:**
```python
spark.sql("""
    SELECT 
        zona,
        SUM(ventas) as total
    FROM ventas
    WHERE ventas > 100000
    GROUP BY zona
    ORDER BY total DESC
""")
```

**Resultado:** IDÉNTICO (mismo plan de ejecución)

---

### 🛠️ Trabajar con Vistas Temporales

**1️⃣ Crear vista desde DataFrame:**
```python
df = spark.table("ventas")
df.createOrReplaceTempView("ventas_view")
```

**2️⃣ Usar en SQL:**
```python
result = spark.sql("SELECT * FROM ventas_view WHERE zona = 'Centro'")
```

**3️⃣ El resultado es un DataFrame:**
```python
result.show()
result.filter(...)
result.groupBy(...)
```

---

### 🪟 Window Functions: Introducción

**Window Functions** operan sobre un "grupo" de filas relacionadas.

**Caso de uso:** Calcular ranking, acumulados, promedios móviles.

**Ejemplo SQL:**
```sql
SELECT 
    sucursal,
    fecha,
    ventas,
    SUM(ventas) OVER (PARTITION BY sucursal ORDER BY fecha) as ventas_acumuladas,
    ROW_NUMBER() OVER (PARTITION BY sucursal ORDER BY ventas DESC) as ranking
FROM ventas
```

**Componentes:**
* `PARTITION BY`: Divide en grupos
* `ORDER BY`: Orden dentro del grupo
* `OVER`: Define la ventana

---

### 🔷 Delta Lake: Almacenamiento ACID

**¿Qué es Delta Lake?**

Formato de almacenamiento que agrega capacidades ACID sobre Parquet.

**Ventajas:**

| Parquet | Delta Lake |
|---------|------------|
| Solo lectura | Lectura + Escritura transaccional |
| Sin versionado | Time Travel (versiones) |
| Sin ACID | Transacciones ACID |
| Sobrescritura completa | Merge/Upsert incremental |

**ACID:**
* **A**tomicity: Todo o nada
* **C**onsistency: Datos consistentes
* **I**solation: Sin interferencias
* **D**urability: Cambios permanentes

---

### 🕐 Time Travel

**Concepto:** Acceder a versiones anteriores de una tabla.

**Sintaxis:**
```python
# Leer versión 5
df = spark.read.format("delta").option("versionAsOf", 5).load("/path/to/table")

# Leer snapshot de ayer
df = spark.read.format("delta").option("timestampAsOf", "2024-01-01").load("/path/to/table")
```

**Casos de uso:**
* 🔙 **Rollback:** Deshacer cambios erróneos
* 📊 **Auditoría:** Ver qué había antes
* 🐛 **Debug:** Comparar versiones

---

### 🔄 Merge/Upsert en Delta

**Problema:** Actualizar registros existentes + insertar nuevos.

**Solución SQL:**
```sql
MERGE INTO target t
USING source s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
```

**Uso:** ETL incremental (CDC - Change Data Capture)

---

### 💡 Cuándo Usar Cada Uno

**DataFrame API:**
* ✅ Pipelines programáticos
* ✅ Lógica compleja en Python
* ✅ Code reuse y testing

**SQL API:**
* ✅ Analistas sin programación
* ✅ Queries ad-hoc
* ✅ Migraciones desde SQL tradicional

**Delta Lake:**
* ✅ Pipelines de producción
* ✅ Datos cambiantes (updates/deletes)
* ✅ Auditoría y compliance

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
import warnings
warnings.filterwarnings('ignore')

print("⚡ PYSPARK SQL, WINDOW Y DELTA LAKE")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    print(f"Versión de Spark: {spark.version}")
except:
    print("⚠️  SparkSession no disponible")

print("\n🎯 En este notebook aprenderás:")
print("  • spark.sql() - Ejecutar SQL distribuido")
print("  • createOrReplaceTempView() - Crear vistas SQL")
print("  • Window Functions - Ranking y acumulados")
print("  • Delta Lake - Formato ACID")
print("  • Time Travel - Versionado de datos")

print("\n📖 Métodos clave:")
print("  - spark.sql('SELECT ...')")
print("  - df.createOrReplaceTempView('nombre')")
print("  - spark.read.format('delta').load(path)")
print("  - df.write.format('delta').save(path)")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

In [0]:
print("Módulo 13: PySpark SQL y tablas Delta Lake en Databricks Free Edition")
print("Soporte nativo para Time Travel y transacciones ACID en el Lakehouse.")

